# MNIST Classification with a NumPy MLP

This notebook implements a one-hidden-layer Multi-Layer Perceptron from scratch using NumPy. It follows the original project logic and adds clearer comments, docstrings, and reproducible paths.


## 1. Imports and Project Settings


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# The notebook is stored in notebooks/, so the dataset is one folder up in data/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "mnist.npz"

print("Dataset path:", DATA_PATH)


## 2. Load the MNIST Dataset


In [ ]:
# Load the MNIST dataset from a local .npz file.
# Expected arrays:
# x_train: training images with shape (60000, 28, 28)
# y_train: training labels with shape (60000,)
# x_test: test images with shape (10000, 28, 28)
# y_test: test labels with shape (10000,)

data = np.load(DATA_PATH)
print("keys:", data.files)

X_train_img = data["x_train"]
y_train = data["y_train"]
X_test_img = data["x_test"]
y_test = data["y_test"]

print("train images:", X_train_img.shape)
print("train labels:", y_train.shape)
print("test images:", X_test_img.shape)
print("test labels:", y_test.shape)


## 3. Data Preprocessing

The images are flattened from 28 x 28 into 784 input features, normalized to the range [0, 1], and labels are converted to one-hot vectors.


In [ ]:
def to_one_hot(labels, num_classes=10):
    """Convert integer class labels into one-hot encoded rows."""
    encoded = np.zeros((labels.shape[0], num_classes), dtype=np.float32)
    encoded[np.arange(labels.shape[0]), labels] = 1.0
    return encoded

# Flatten images and normalize pixel values.
X_train = X_train_img.reshape(-1, 28 * 28).astype(np.float32) / 255.0
X_test = X_test_img.reshape(-1, 28 * 28).astype(np.float32) / 255.0

Y_train = to_one_hot(y_train, num_classes=10)
Y_test = to_one_hot(y_test, num_classes=10)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)


## 4. Neural Network Building Blocks


In [ ]:
class Linear:
    """Fully connected layer: computes X @ W + b and stores gradients."""

    def __init__(self, in_dim, out_dim, weight_scale=0.01):
        self.W = (np.random.randn(in_dim, out_dim) * weight_scale).astype(np.float32)
        self.b = np.zeros((1, out_dim), dtype=np.float32)

        # Cached input for the backward pass.
        self.X = None

        # Parameter gradients.
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        """Run the forward pass for the linear layer."""
        self.X = X
        return X @ self.W + self.b

    def backward(self, dZ):
        """Compute gradients and return the gradient with respect to the input."""
        self.dW = self.X.T @ dZ
        self.db = np.sum(dZ, axis=0, keepdims=True)
        return dZ @ self.W.T

    def step(self, lr):
        """Update weights and biases using gradient descent."""
        self.W -= lr * self.dW
        self.b -= lr * self.db


class ReLU:
    """ReLU activation for the hidden layer."""

    def __init__(self):
        self.mask = None

    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask

    def backward(self, dA):
        return dA * self.mask


class Sigmoid:
    """Sigmoid activation for the output layer."""

    def __init__(self):
        self.out = None

    def forward(self, Z):
        self.out = 1.0 / (1.0 + np.exp(-Z))
        return self.out

    def backward(self, dA):
        return dA * self.out * (1.0 - self.out)


class MSELoss:
    """Mean Squared Error loss used with one-hot labels."""

    def __init__(self):
        self.P = None
        self.Y = None

    def forward(self, P, Y):
        self.P = P
        self.Y = Y
        return np.mean((P - Y) ** 2)

    def backward(self):
        batch_size = self.Y.shape[0]
        return (2.0 * (self.P - self.Y)) / batch_size


## 5. One-Hidden-Layer MLP


In [ ]:
class MLP1Hidden:
    """A simple MLP with one hidden layer implemented from scratch."""

    def __init__(self, input_dim=784, hidden_dim=128, num_classes=10):
        self.fc1 = Linear(input_dim, hidden_dim, weight_scale=0.01)
        self.act1 = ReLU()

        self.fc2 = Linear(hidden_dim, num_classes, weight_scale=0.01)
        self.act2 = Sigmoid()

        self.loss_fn = MSELoss()

    def forward(self, X, Y=None):
        """Return predictions, or loss when labels are provided."""
        z1 = self.fc1.forward(X)
        a1 = self.act1.forward(z1)

        z2 = self.fc2.forward(a1)
        p = self.act2.forward(z2)

        if Y is None:
            return p
        return self.loss_fn.forward(p, Y)

    def backward(self):
        """Backpropagate through loss, output layer, hidden activation, and input layer."""
        dP = self.loss_fn.backward()
        dZ2 = self.act2.backward(dP)
        dA1 = self.fc2.backward(dZ2)
        dZ1 = self.act1.backward(dA1)
        self.fc1.backward(dZ1)

    def step(self, lr):
        self.fc1.step(lr)
        self.fc2.step(lr)

    def predict(self, X):
        probabilities = self.forward(X, Y=None)
        return np.argmax(probabilities, axis=1)


## 6. Training Utilities


In [ ]:
def accuracy(y_true, y_pred):
    """Calculate classification accuracy."""
    return np.mean(y_true == y_pred)


def iterate_minibatches(X, Y, batch_size=128, shuffle=True):
    """Yield mini-batches from X and Y."""
    num_samples = X.shape[0]
    indices = np.arange(num_samples)

    if shuffle:
        np.random.shuffle(indices)

    for start in range(0, num_samples, batch_size):
        batch_indices = indices[start:start + batch_size]
        yield X[batch_indices], Y[batch_indices]


def train(model, lr=0.1, epochs=10, batch_size=128, train_acc_subset=10000):
    """Train the model and store loss, accuracy, and epoch time history."""
    history = {"loss": [], "train_acc": [], "test_acc": [], "time": []}

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        losses = []

        for X_batch, Y_batch in iterate_minibatches(X_train, Y_train, batch_size=batch_size, shuffle=True):
            loss = model.forward(X_batch, Y_batch)
            model.backward()
            model.step(lr)
            losses.append(loss)

        train_pred = model.predict(X_train[:train_acc_subset])
        test_pred = model.predict(X_test)

        train_acc = accuracy(y_train[:train_acc_subset], train_pred)
        test_acc = accuracy(y_test, test_pred)

        history["loss"].append(float(np.mean(losses)))
        history["train_acc"].append(float(train_acc))
        history["test_acc"].append(float(test_acc))
        history["time"].append(float(time.time() - start_time))

        print(
            f"Epoch {epoch:02d}  loss={history['loss'][-1]:.4f}  "
            f"train_acc={train_acc:.4f}  test_acc={test_acc:.4f}  "
            f"time={history['time'][-1]:.2f}s"
        )

    return history


## 7. Question 1: Train the Base Model

The original notebook used `lr=0.5` for this run. The written report mentions `lr=0.1`, so keep the value consistent with the result you want to reproduce.


In [ ]:
hidden_dim = 128
learning_rate = 0.5
model = MLP1Hidden(hidden_dim=hidden_dim)
hist = train(model, lr=learning_rate, epochs=10, batch_size=128)


## 8. Plot Training Loss and Accuracy


In [ ]:
plt.figure()
plt.plot(hist["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training Loss (MSE)")
plt.show()

plt.figure()
plt.plot(hist["train_acc"], label="Train Accuracy")
plt.plot(hist["test_acc"], label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()
plt.show()


## 9. Show Sample Predictions


In [ ]:
def show_predictions(model, n=12):
    """Display random test images with true and predicted labels."""
    sample_indices = np.random.choice(X_test.shape[0], size=n, replace=False)
    predictions = model.predict(X_test[sample_indices])

    plt.figure(figsize=(12, 4))
    for i in range(n):
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(X_test_img[sample_indices[i]], cmap="gray")
        plt.axis("off")
        plt.title(f"y={y_test[sample_indices[i]]}, p={predictions[i]}")
    plt.tight_layout()
    plt.show()

show_predictions(model, n=12)


## 10. Parameter Summary


In [ ]:
def parameter_summary(array):
    """Return simple descriptive statistics for a parameter array."""
    return {
        "shape": array.shape,
        "mean": float(np.mean(array)),
        "var": float(np.var(array)),
        "min": float(np.min(array)),
        "max": float(np.max(array)),
    }

summary = {
    "W1": parameter_summary(model.fc1.W),
    "b1": parameter_summary(model.fc1.b),
    "W2": parameter_summary(model.fc2.W),
    "b2": parameter_summary(model.fc2.b),
}
summary


## 11. Question 2: Visualize Hidden Neuron Sensitivity


In [ ]:
def show_interesting_hidden_neurons(model, num_neurons=8):
    """Visualize hidden neurons with the largest input-weight variance."""
    first_layer_weights = model.fc1.W

    # A high variance means the neuron gives very different weights to image pixels.
    scores = np.var(first_layer_weights, axis=0)
    top_indices = np.argsort(scores)[-num_neurons:][::-1]

    columns = 4
    rows = int(np.ceil(num_neurons / columns))
    plt.figure(figsize=(4 * columns, 4 * rows))

    for i, neuron_index in enumerate(top_indices):
        weight_image = first_layer_weights[:, neuron_index].reshape(28, 28)
        plt.subplot(rows, columns, i + 1)
        plt.imshow(weight_image, cmap="gray")
        plt.axis("off")
        plt.title(f"Neuron {neuron_index}")

    plt.tight_layout()
    plt.show()

show_interesting_hidden_neurons(model, num_neurons=8)


## 12. Question 3: Compare Different Hidden Layer Sizes


In [ ]:
configs = [64, 128, 256]
results = {}

for hidden_units in configs:
    print("\n" + "=" * 60)
    print(f"Training with hidden_dim={hidden_units}")
    current_model = MLP1Hidden(hidden_dim=hidden_units)
    current_history = train(current_model, lr=0.1, epochs=10, batch_size=128)
    results[hidden_units] = {"model": current_model, "hist": current_history}


In [ ]:
plt.figure()
for hidden_units in configs:
    plt.plot(results[hidden_units]["hist"]["test_acc"], label=f"H={hidden_units}")
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy")
plt.title("Test Accuracy Comparison")
plt.legend()
plt.show()


In [ ]:
for hidden_units in configs:
    mean_epoch_time = np.mean(results[hidden_units]["hist"]["time"])
    best_test_accuracy = np.max(results[hidden_units]["hist"]["test_acc"])
    print(
        f"H={hidden_units:>3}  "
        f"mean_epoch_time={mean_epoch_time:.2f}s  "
        f"best_test_acc={best_test_accuracy:.4f}"
    )
